Reference:https://towardsdatascience.com/augmenting-llms-with-rag-f79de914e672

In [1]:
import langchain
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.embeddings.cache import CacheBackedEmbeddings
from langchain.vectorstores import FAISS
from langchain.storage import LocalFileStore
from langchain.document_loaders import PyPDFLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

In [2]:
from dotenv import load_dotenv
 
load_dotenv()

True

RAG Setup

We need the following to implement RAG with LangChain:

* Storage: A local store for our data we are providing for our RAG application. To scale up you can utilize other stores such as S3 as this gets larger
* Embeddings model: To create embedding out of the provided data, we use OpenAI Embeddings
* Vector Store: Store model embeddings, FAISS in this case
* Chain: Stitches together these different components, our LLM models is OpenAI in this case

In [5]:
# where our embeddings will be stored
store = LocalFileStore("./cache")

In [6]:
# instantiate a loader. If there are multiple pdf, and we have to load them together then use PyPDFDirectoryLoader
loader = PyPDFLoader("Enqurious_ETL_DB document.pdf")

In [7]:
pages = loader.load_and_split()

In [8]:
print(len(pages))

36


In [9]:
# instantiate the embeddings model
embedding_model = OpenAIEmbeddings()

In [10]:
# pass in our vector store
embedder = CacheBackedEmbeddings.from_bytes_store(
    embedding_model,
    store
)

Reference: https://python.langchain.com/docs/modules/data_connection/text_embedding/caching_embeddings

CacheBackedEmbeddings will store the embeddings in our Local storage in a folder called ./cache via object that we created above: ```store = LocalFileStore("./cache")```. We can give other name also.

It is good to store it on cloud

FAISS REFERNCE: https://ai.meta.com/tools/faiss/#:~:text=FAISS%20(Facebook%20AI%20Similarity%20Search,more%20scalable%20similarity%20search%20functions.

In [19]:
# !pip install faiss-cpu

     ---------------------------------------- 10.8/10.8 MB 5.6 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3.1 -> 23.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# pass in our vector store
vector_store = FAISS.from_documents(
    pages,
    embedder
)

In [12]:
etl_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(),
    # chain_type="stuff", # by default chain type is stuff only, so no need to mention it explicitly
    retriever=vector_store.as_retriever(),
    return_source_documents=True,
    verbose=True
)

In [13]:
import openai

In [14]:
prompt = "How many tables are there in ETL database?"

In [15]:
#vanilla OpenAI Response, without RAG
response = openai.Completion.create(
    engine="text-davinci-003",
    prompt=prompt,
    max_tokens=500
)

In [16]:
print(response["choices"][0]["text"])



There is no definitive answer to this question as the number of tables in an ETL database will depend on the needs and scope of the specific project.


As it is observed that before embedding, the model doesn't have any definitive answer for our question.

In [17]:
# The response after using embeddings
response_rag = etl_chain({"query": prompt})



> Entering new RetrievalQA chain...

> Finished chain.


In [18]:
response_rag

{'query': 'How many tables are there in ETL database?',
 'result': 'Based on the provided context, the number of tables in the ETL database is not specified. Therefore, I do not know the exact number of tables in the ETL database.',
 'source_documents': [Document(page_content='Data Loading:  \nIn the world of data processing, ETL (Extract, Transform, Load) process played a crucial role in \norganizing and managing data. The final step in this process is the Loading phase, where the \nextracted and transformed dat a found their new home in a target database.  \nTo ensure efficient data loading, a few steps were taken. First, the database needed to be properly \nindexed to optimize query performance. Additionally, constraints were temporarily disabled during \nthe loading process to facilitate smooth data insertion. I t was important to run all three steps of the \nETL process in parallel to save time. While data extraction took place, the transformation phase \nwould begin simultaneousl

In [19]:
# Give clear instructions to find the output
prompt2 = "Refer the ETL Data Dictionary and find how many tables are there in ETL database?"

> Now let's ask this question to model with no context

In [21]:
response = openai.Completion.create(
    engine="text-davinci-003",
    prompt=prompt2,
    max_tokens=500
)

In [22]:
print(response["choices"][0]["text"])



The number of tables in an ETL database will vary depending on the specific ETL requirements. Generally speaking, it is likely to include at least one source table, one target table, one staging table (or several, depending on the ETL requirements), and possibly other related tables such as lookup tables, control tables, audit tables, etc.


> Now provide the same question to model with context

In [23]:
# The response after using embeddings
response_rag = etl_chain({"query": prompt2})



> Entering new RetrievalQA chain...

> Finished chain.


In [24]:
response_rag['result']

'Based on the provided information from the ETL Data Dictionary, there are a total of 6 tables in the ETL database.'

> We can observe that the model having context have successfully fetched the correct number of tables from the document

In [35]:
prompt3 = "Refer the ETL Data Dictionary and return the names of tables present in ETL database?"

In [38]:
response_rag = etl_chain({"query": prompt3})



> Entering new RetrievalQA chain...

> Finished chain.


In [40]:
print(response_rag['result'])

Based on the provided ETL Data Dictionary, the names of the tables present in the ETL database are:

1. clients
2. progress_fact
3. skills_fact
4. feedback_fact
5. learners_fact
6. order_duration_fact


In [42]:
print(etl_chain({"query": 'What is the primary key of clients table?'})['result'])



> Entering new RetrievalQA chain...

> Finished chain.
Based on the given context, the primary key of the clients table is the "client_id" column.


In [43]:
print(etl_chain({"query": 'What is the primary key of progress_fact table?'})['result'])



> Entering new RetrievalQA chain...

> Finished chain.
The primary key of the progress_fact table is the "id" column.


In [44]:
# Now ask the model about the granularity of the progress_fact table
response_rag = etl_chain({"query": 'What is the granularity level of the progress_fact table?'})
response_rag['result']



> Entering new RetrievalQA chain...

> Finished chain.


'The granularity level of the progress_fact table is at the activity level. Each row in the table represents the progress of a learner in a specific activity.'

In [45]:
# Now ask the model about the granularity of the progress_fact table
response_rag = etl_chain({"query": 'What is the granularity level of the skills_fact table?'})
response_rag['result']



> Entering new RetrievalQA chain...

> Finished chain.


'The granularity level of the skills_fact table is at the activity level. Skills are grouped at the activity level, meaning that if a skill is linked to multiple inputs within a particular activity, there will be only one record for that skill in the skills_fact table.'

The responses given by the embedded model is correct to an extent.

In [47]:
etl_chain({"query": 'Refer the data dictionary of Enqurious ETL document and frame the sql query for my question based on that. So how can we find top 5 learners of the Excel Essentials based on scores from the database?'})



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Refer the data dictionary of Enqurious ETL document and frame the sql query for my question based on that. So how can we find top 5 learners of the Excel Essentials based on scores from the database?',
 'result': "To find the top 5 learners of the Excel Essentials based on scores from the database, you can use the following SQL query:\n\n```sql\nSELECT learners_fact.learner_fact_id, learners_fact.order_id, learners_fact.client_id, learners_fact.score\nFROM learners_fact\nJOIN skills_fact ON learners_fact.skill_fact_id = skills_fact.skill_fact_id\nWHERE skills_fact.skill_name = 'Excel Essentials'\nORDER BY learners_fact.score DESC\nLIMIT 5;\n```\n\nThis query selects the necessary columns from the `learners_fact` table and joins it with the `skills_fact` table using the `skill_fact_id` column. It then filters the records based on the skill name 'Excel Essentials'. The results are sorted in descending order by the learner's score and limited to the top 5 learners.",
 'source_d

In [48]:
# Have copied the output from above and printed here
print("To find the top 5 learners of the Excel Essentials based on scores from the database, you can use the following SQL query:\n\n```sql\nSELECT learners_fact.learner_fact_id, learners_fact.order_id, learners_fact.client_id, learners_fact.score\nFROM learners_fact\nJOIN skills_fact ON learners_fact.skill_fact_id = skills_fact.skill_fact_id\nWHERE skills_fact.skill_name = 'Excel Essentials'\nORDER BY learners_fact.score DESC\nLIMIT 5;\n```\n\nThis query selects the necessary columns from the `learners_fact` table and joins it with the `skills_fact` table using the `skill_fact_id` column. It then filters the records based on the skill name 'Excel Essentials'. The results are sorted in descending order by the learner's score and limited to the top 5 learners.")

To find the top 5 learners of the Excel Essentials based on scores from the database, you can use the following SQL query:

```sql
SELECT learners_fact.learner_fact_id, learners_fact.order_id, learners_fact.client_id, learners_fact.score
FROM learners_fact
JOIN skills_fact ON learners_fact.skill_fact_id = skills_fact.skill_fact_id
WHERE skills_fact.skill_name = 'Excel Essentials'
ORDER BY learners_fact.score DESC
LIMIT 5;
```

This query selects the necessary columns from the `learners_fact` table and joins it with the `skills_fact` table using the `skill_fact_id` column. It then filters the records based on the skill name 'Excel Essentials'. The results are sorted in descending order by the learner's score and limited to the top 5 learners.


The above query is incorrect

In [25]:
response_rag =  etl_chain({"query": "Which attribute represents the scores of the learners in skills_fact table?"})
response_rag



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Which attribute represents the scores of the learners in skills_fact table?',
 'result': 'The "score" attribute in the skills_fact table represents the scores of the learners in each skill.',
 'source_documents': [Document(page_content='for a consolidated view of \nthe learner\'s performance in \neach skill at the activity \nlevel.  \n39 skill_fact  total INTEGER  The "total" attribute in the \ndata represents the total \nmarks possible for a \nparticular skill in a specific \nactivity. It indicates the \nmaximum score that can be \nachieved for that skill in the \ngiven activity.  The "total" attribute in the data \nindicates the t otal marks or points \nassociated with a particular skill in a \nspecific activity. The value of "total" \nshould be greater than or equal to 0. \nFor example, if a skill with a fixed total \nof 10 marks is attached to five different \ninputs within a particular activ ity, the \ntotal for that skill at the activity level \nwould be 50.  \n40 skil

In [26]:
print(response_rag['result'])

The "score" attribute in the skills_fact table represents the scores of the learners in each skill.


> Based on he above conversation it can be concluded that the theoretical questions are properly answered by the embedded model, but the sql query are not properly fetched out from that.